# GVH Diagonal Cubic 0.3.1.2 — Real Observation Data Framework: Theory–Data Gates

**Partie C — Physical Predictions and Falsification**

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.1.2  
**Statut :** consolidation post-Articles I–II  
**But :** séparer strictement la préparation des données de la préparation théorique d’un test GVH.

---

## Principe central

Les Articles I et II imposent désormais une règle méthodologique supplémentaire :

> **Une donnée peut être prête pour l’analyse sans que GVH possède encore une prédiction indépendante à tester.**

Ce notebook introduit donc deux portes indépendantes :

\[
\mathrm{DATA\_READY}
\]

et

\[
\mathrm{THEORY\_READY}.
\]

Un test GVH n’est autorisé que si

\[
\boxed{
\mathrm{READY\_FOR\_GVH\_TEST}
=
\mathrm{DATA\_READY}
\land
\mathrm{THEORY\_READY}
}
\]

Cette version ne crée aucune nouvelle prédiction physique. Elle empêche au contraire qu’une compatibilité avec GR, une hypothèse ou un fit aux données soit automatiquement présenté comme une prédiction GVH indépendante.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_ID = "GVH_Diagonal_Cubic_0.3.1.2"
NOTEBOOK_VERSION = "0.3.1.2"
PART_C_VERSION = "0.2C"
FRAMEWORK_NAME = "GVH Diagonal Cubic"
AUTHOR = "Charlemagne O Laurince"
EXECUTION_UTC = datetime.now(timezone.utc).isoformat()

ENVIRONMENT_INFO = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "part_c_version": PART_C_VERSION,
    "framework_name": FRAMEWORK_NAME,
    "author": AUTHOR,
    "execution_utc": EXECUTION_UTC,
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "operating_system": os.name,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
}

pd.DataFrame([ENVIRONMENT_INFO]).T.rename(columns={0: "value"})


,value
notebook_id,GVH_Diagonal_Cubic_0.3.1.2
notebook_version,0.3.1.2
part_c_version,0.2C
framework_name,GVH Diagonal Cubic
author,Charlemagne O Laurince
execution_utc,2026-08-07T23:20:32.915136+00:00
python_version,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11...."
python_executable,/usr/bin/python3
platform,Linux-6.6.122+-x86_64-with-glibc2.35
operating_system,posix


# 1. Arborescence reproductible

La structure de 0.3.1.1 est conservée. Les données brutes restent séparées des données extraites, transformées et préparées.

```text
gvh_diagonal_cubic/
├── data/
│   ├── raw/
│   ├── extracted/
│   ├── processed/
│   ├── prepared/
│   ├── metadata/
│   └── checksums/
└── exports/
```


In [2]:
PROJECT_ROOT = Path(".")
DATA_ROOT = PROJECT_ROOT / "data"

RAW_DIR = DATA_ROOT / "raw"
EXTRACTED_DIR = DATA_ROOT / "extracted"
PROCESSED_DIR = DATA_ROOT / "processed"
PREPARED_DIR = DATA_ROOT / "prepared"
METADATA_DIR = DATA_ROOT / "metadata"
CHECKSUM_DIR = DATA_ROOT / "checksums"
EXPORT_DIR = PROJECT_ROOT / "exports"

DIRECTORIES = [
    RAW_DIR, EXTRACTED_DIR, PROCESSED_DIR, PREPARED_DIR,
    METADATA_DIR, CHECKSUM_DIR, EXPORT_DIR,
]

for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

directory_status_df = pd.DataFrame([
    {
        "directory": str(directory),
        "exists": directory.exists(),
        "is_directory": directory.is_dir(),
    }
    for directory in DIRECTORIES
])

directory_status_df


,directory,exists,is_directory
0,data/raw,True,True
1,data/extracted,True,True
2,data/processed,True,True
3,data/prepared,True,True
4,data/metadata,True,True
5,data/checksums,True,True
6,exports,True,True


# 2. Registre des données

Les statuts de données décrivent uniquement la maturité des fichiers et métadonnées. Ils ne doivent jamais être interprétés comme un verdict physique sur GVH.


In [3]:
ALLOWED_DATA_STATUSES = {
    "PLANNED",
    "NOT_LOADED",
    "DOWNLOADED",
    "INTEGRITY_VERIFIED",
    "SCHEMA_VALIDATED",
    "READY_FOR_ANALYSIS",
    "REJECTED",
}

@dataclass
class DatasetRecord:
    dataset_id: str
    domain: str
    title: str
    source_organization: str
    official_reference: str
    version: str = "UNSPECIFIED"
    access_date_utc: str = "NOT_ACCESSED"
    license_name: str = "UNVERIFIED"
    citation_text: str = "TO_BE_COMPLETED"
    expected_format: str = "UNKNOWN"
    raw_relative_path: str = ""
    prepared_relative_path: str = ""
    expected_columns: list = field(default_factory=list)
    expected_units: dict = field(default_factory=dict)
    covariance_required: bool = False
    checksum_sha256: str = ""
    status: str = "NOT_LOADED"
    notes: str = ""

    def validate(self):
        if self.status not in ALLOWED_DATA_STATUSES:
            raise ValueError(f"Statut de données invalide: {self.status}")
        if not self.dataset_id:
            raise ValueError("dataset_id vide")
        return True


In [4]:
DATASET_REGISTRY = [
    DatasetRecord(
        "EOS_NUCLEAR_TABLES",
        "neutron_star_eos",
        "Verified nuclear EOS tables",
        "MULTIPLE — TO VERIFY",
        "TO_BE_VERIFIED",
        expected_format="CSV/TXT/HDF5",
        raw_relative_path="data/raw/eos/",
        prepared_relative_path="data/prepared/eos/",
        expected_columns=["energy_density", "pressure", "baryon_density"],
        expected_units={
            "energy_density": "MUST_BE_DOCUMENTED",
            "pressure": "MUST_BE_DOCUMENTED",
            "baryon_density": "MUST_BE_DOCUMENTED",
        },
        notes="Une provenance, une version et une licence par EOS.",
    ),
    DatasetRecord(
        "NICER_MASS_RADIUS",
        "neutron_star_structure",
        "NICER mass-radius posterior products",
        "NICER collaboration / official archive",
        "TO_BE_VERIFIED",
        expected_format="HDF5/FITS/CSV",
        raw_relative_path="data/raw/nicer/",
        prepared_relative_path="data/prepared/nicer/",
        expected_columns=["mass_solar", "radius_km", "posterior_weight"],
        expected_units={
            "mass_solar": "solar_mass",
            "radius_km": "km",
            "posterior_weight": "dimensionless",
        },
        covariance_required=True,
    ),
    DatasetRecord(
        "MASSIVE_PULSARS",
        "neutron_star_structure",
        "High-mass pulsar measurements",
        "Published timing collaborations",
        "TO_BE_VERIFIED",
        expected_format="CSV",
        raw_relative_path="data/raw/pulsars/",
        prepared_relative_path="data/prepared/pulsars/",
        expected_columns=["source_name", "mass_solar", "mass_sigma_solar"],
        expected_units={
            "mass_solar": "solar_mass",
            "mass_sigma_solar": "solar_mass",
        },
    ),
    DatasetRecord(
        "GW_PUBLIC_STRAIN",
        "gravitational_waves",
        "Public calibrated GW strain",
        "Official open GW archive",
        "TO_BE_VERIFIED",
        expected_format="HDF5/GWF",
        raw_relative_path="data/raw/gw/strain/",
        prepared_relative_path="data/prepared/gw/strain/",
        expected_columns=["time", "strain"],
        expected_units={"time": "s", "strain": "dimensionless"},
    ),
    DatasetRecord(
        "GW_EVENT_POSTERIORS",
        "gravitational_waves",
        "Public compact-binary posterior samples",
        "Official collaboration data release",
        "TO_BE_VERIFIED",
        expected_format="HDF5/JSON/CSV",
        raw_relative_path="data/raw/gw/posteriors/",
        prepared_relative_path="data/prepared/gw/posteriors/",
        expected_columns=["event_id", "parameter", "value"],
        expected_units={"event_id": "dimensionless", "parameter": "mixed", "value": "observable_specific"},
    ),
    DatasetRecord(
        "PPN_SOLAR_SYSTEM",
        "weak_field",
        "Solar-system PPN constraints",
        "Published / official experimental sources",
        "TO_BE_VERIFIED",
        expected_format="CSV",
        raw_relative_path="data/raw/ppn/",
        prepared_relative_path="data/prepared/ppn/",
        expected_columns=["parameter", "central_value", "sigma", "reference"],
        expected_units={
            "parameter": "dimensionless",
            "central_value": "dimensionless",
            "sigma": "dimensionless",
            "reference": "dimensionless",
        },
    ),
    DatasetRecord(
        "PANTHEON_PLUS",
        "cosmology",
        "Pantheon+ supernova compilation",
        "Official collaboration release",
        "TO_BE_VERIFIED",
        expected_format="CSV/TXT",
        raw_relative_path="data/raw/cosmology/pantheon_plus/",
        prepared_relative_path="data/prepared/cosmology/pantheon_plus/",
    ),
    DatasetRecord(
        "BAO_COMPILATION",
        "cosmology",
        "BAO measurements",
        "Published collaboration releases",
        "TO_BE_VERIFIED",
        expected_format="CSV/TXT",
        raw_relative_path="data/raw/cosmology/bao/",
        prepared_relative_path="data/prepared/cosmology/bao/",
    ),
    DatasetRecord(
        "HUBBLE_HZ",
        "cosmology",
        "H(z) measurements",
        "Published compilation / original sources",
        "TO_BE_VERIFIED",
        expected_format="CSV",
        raw_relative_path="data/raw/cosmology/hz/",
        prepared_relative_path="data/prepared/cosmology/hz/",
    ),
    DatasetRecord(
        "CMB_PRODUCTS",
        "cosmology",
        "Public CMB products",
        "Official collaboration archive",
        "TO_BE_VERIFIED",
        expected_format="FITS",
        raw_relative_path="data/raw/cosmology/cmb/",
        prepared_relative_path="data/prepared/cosmology/cmb/",
    ),
]

for record in DATASET_REGISTRY:
    record.validate()

registry_df = pd.DataFrame([asdict(record) for record in DATASET_REGISTRY])
registry_df[["dataset_id", "domain", "status"]]


,dataset_id,domain,status
0,EOS_NUCLEAR_TABLES,neutron_star_eos,NOT_LOADED
1,NICER_MASS_RADIUS,neutron_star_structure,NOT_LOADED
2,MASSIVE_PULSARS,neutron_star_structure,NOT_LOADED
3,GW_PUBLIC_STRAIN,gravitational_waves,NOT_LOADED
4,GW_EVENT_POSTERIORS,gravitational_waves,NOT_LOADED
5,PPN_SOLAR_SYSTEM,weak_field,NOT_LOADED
6,PANTHEON_PLUS,cosmology,NOT_LOADED
7,BAO_COMPILATION,cosmology,NOT_LOADED
8,HUBBLE_HZ,cosmology,NOT_LOADED
9,CMB_PRODUCTS,cosmology,NOT_LOADED


# 3. Porte DATA_READY

Cette porte répond uniquement à la question :

> **Les données sont-elles suffisamment documentées, vérifiées et structurées pour une analyse reproductible ?**


In [5]:
def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(path)

    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_local_files(record, project_root=PROJECT_ROOT):
    path = project_root / record.raw_relative_path
    if not path.exists():
        return []
    if path.is_file():
        return [path]
    return sorted(
        candidate
        for candidate in path.rglob("*")
        if candidate.is_file() and candidate.name != ".gitkeep"
    )


def validate_tabular_schema(table, expected_columns):
    missing = sorted(set(expected_columns) - set(table.columns))
    numeric = table.select_dtypes(include=[np.number]).columns
    return {
        "row_count": len(table),
        "missing_columns": missing,
        "extra_columns": sorted(set(table.columns) - set(expected_columns)),
        "duplicate_rows": int(table.duplicated().sum()),
        "null_counts": table.isna().sum().astype(int).to_dict(),
        "nonfinite_counts": {
            c: int((~np.isfinite(table[c].to_numpy(float))).sum())
            for c in numeric
        },
        "schema_pass": len(missing) == 0 and len(table) > 0,
    }


REQUIRED_METADATA_FIELDS = [
    "dataset_id", "title", "source_organization", "official_reference",
    "version", "access_date_utc", "license_name", "citation_text",
    "expected_format", "status",
]

PLACEHOLDERS = {
    "", "UNSPECIFIED", "UNVERIFIED", "TO_BE_COMPLETED",
    "TO_BE_VERIFIED", "NOT_ACCESSED",
}


def validate_record_metadata(record):
    data = asdict(record)
    missing = [
        field
        for field in REQUIRED_METADATA_FIELDS
        if data.get(field) is None or str(data.get(field)) in PLACEHOLDERS
    ]
    return {
        "dataset_id": record.dataset_id,
        "metadata_ready": not missing,
        "missing_or_placeholder": missing,
    }


def data_readiness_gate(
    file_present,
    checksum_recorded,
    schema_validated,
    units_documented,
    license_verified,
    metadata_ready,
):
    checks = {
        "file_present": bool(file_present),
        "checksum_recorded": bool(checksum_recorded),
        "schema_validated": bool(schema_validated),
        "units_documented": bool(units_documented),
        "license_verified": bool(license_verified),
        "metadata_ready": bool(metadata_ready),
    }
    checks["DATA_READY"] = all(checks.values())
    return checks


In [6]:
metadata_audit_df = pd.DataFrame([
    validate_record_metadata(record)
    for record in DATASET_REGISTRY
])

availability_rows = []

for record in DATASET_REGISTRY:
    files = find_local_files(record)
    meta = validate_record_metadata(record)

    units_documented = bool(record.expected_units) and all(
        value not in {"", "MUST_BE_DOCUMENTED", "observable_specific", "mixed"}
        for value in record.expected_units.values()
    )

    gate = data_readiness_gate(
        file_present=bool(files),
        checksum_recorded=bool(record.checksum_sha256),
        schema_validated=record.status in {"SCHEMA_VALIDATED", "READY_FOR_ANALYSIS"},
        units_documented=units_documented,
        license_verified=record.license_name not in {"", "UNVERIFIED"},
        metadata_ready=meta["metadata_ready"],
    )

    availability_rows.append({
        "dataset_id": record.dataset_id,
        "domain": record.domain,
        "status": record.status,
        "file_count": len(files),
        **gate,
    })

availability_df = pd.DataFrame(availability_rows)
availability_df


,dataset_id,domain,status,file_count,file_present,checksum_recorded,schema_validated,units_documented,license_verified,metadata_ready,DATA_READY
0,EOS_NUCLEAR_TABLES,neutron_star_eos,NOT_LOADED,0,False,False,False,False,False,False,False
1,NICER_MASS_RADIUS,neutron_star_structure,NOT_LOADED,0,False,False,False,True,False,False,False
2,MASSIVE_PULSARS,neutron_star_structure,NOT_LOADED,0,False,False,False,True,False,False,False
3,GW_PUBLIC_STRAIN,gravitational_waves,NOT_LOADED,0,False,False,False,True,False,False,False
4,GW_EVENT_POSTERIORS,gravitational_waves,NOT_LOADED,0,False,False,False,False,False,False,False
5,PPN_SOLAR_SYSTEM,weak_field,NOT_LOADED,0,False,False,False,True,False,False,False
6,PANTHEON_PLUS,cosmology,NOT_LOADED,0,False,False,False,False,False,False,False
7,BAO_COMPILATION,cosmology,NOT_LOADED,0,False,False,False,False,False,False,False
8,HUBBLE_HZ,cosmology,NOT_LOADED,0,False,False,False,False,False,False,False
9,CMB_PRODUCTS,cosmology,NOT_LOADED,0,False,False,False,False,False,False,False


# 4. Registre scientifique post-Articles I–II

Chaque objet utilisé dans un test doit avoir un **statut épistémique explicite**.

Les catégories autorisées sont :

- `DERIVED` : conséquence mathématique dérivée des postulats/équations GVH ;
- `HYPOTHESIS` : hypothèse supplémentaire non encore dérivée ;
- `REFERENCE_GR` : prédiction ou valeur issue de GR servant de référence ;
- `FIT` : quantité déterminée par ajustement aux données ;
- `TESTABLE_GVH` : prédiction GVH indépendante, fixée avant confrontation aux données de test ;
- `INCONCLUSIVE` : information insuffisante pour produire un test physique.

Une relation qui ne fait que reproduire GR reste `REFERENCE_GR` ou `DERIVED_COMPATIBILITY`, mais **ne devient pas automatiquement `TESTABLE_GVH`**.


In [7]:
ALLOWED_THEORY_STATUSES = {
    "DERIVED",
    "DERIVED_COMPATIBILITY",
    "HYPOTHESIS",
    "REFERENCE_GR",
    "FIT",
    "TESTABLE_GVH",
    "INCONCLUSIVE",
    "REJECTED",
}

@dataclass
class TheoryClaim:
    claim_id: str
    domain: str
    observable: str
    mathematical_expression: str
    status: str
    source_notebook: str
    source_type: str
    independent_of_test_data: bool
    parameters_fixed_before_test: bool
    gr_reference_defined: bool
    uncertainty_model_defined: bool
    falsification_rule_defined: bool
    notes: str = ""

    def validate(self):
        if self.status not in ALLOWED_THEORY_STATUSES:
            raise ValueError(f"Statut théorique invalide: {self.status}")
        if not self.claim_id:
            raise ValueError("claim_id vide")
        return True


## Claims hérités de l’état actuel

Les entrées ci-dessous sont volontairement prudentes. Elles n’inventent aucune nouvelle prédiction.

- La carte weak-field vers les paramètres PPN est enregistrée comme **compatibilité dérivée**, pas comme nouvelle prédiction non-GR.
- Le projecteur STF est enregistré comme **objet mathématique dérivé**.
- L’origine/dynamique unique du champ timelike reste **hypothétique / non fermée**.


In [8]:
THEORY_REGISTRY = [
    TheoryClaim(
        claim_id="PPN_MAPPING_GVH",
        domain="weak_field",
        observable="PPN_gamma_beta",
        mathematical_expression="gamma_GVH=b1/a1 ; beta_GVH=a2/a1^2",
        status="DERIVED_COMPATIBILITY",
        source_notebook="GVH_Diagonal_Cubic_0.2.23.x",
        source_type="Article II / weak-field chain",
        independent_of_test_data=True,
        parameters_fixed_before_test=False,
        gr_reference_defined=True,
        uncertainty_model_defined=False,
        falsification_rule_defined=False,
        notes=(
            "Carte formelle vers PPN. Aucune valeur numérique indépendante non-GR "
            "n'est encore fixée par la théorie."
        ),
    ),
    TheoryClaim(
        claim_id="STF_SOURCE_PROJECTOR",
        domain="weak_field",
        observable="anisotropic_stress_projection",
        mathematical_expression="pi_mn=P_mn^ab T_ab",
        status="DERIVED",
        source_notebook="GVH_Diagonal_Cubic_0.2.23.2",
        source_type="Article II / covariant source projector",
        independent_of_test_data=True,
        parameters_fixed_before_test=True,
        gr_reference_defined=False,
        uncertainty_model_defined=False,
        falsification_rule_defined=False,
        notes="Objet mathématique utile, mais pas à lui seul une prédiction observationnelle.",
    ),
    TheoryClaim(
        claim_id="TIMELIKE_FIELD_ORIGIN",
        domain="weak_field",
        observable="u_mu_physical_origin",
        mathematical_expression="NOT_UNIQUELY_DERIVED",
        status="HYPOTHESIS",
        source_notebook="GVH_Diagonal_Cubic_0.2.23.5.1",
        source_type="Article II / physical-origin audit",
        independent_of_test_data=True,
        parameters_fixed_before_test=False,
        gr_reference_defined=False,
        uncertainty_model_defined=False,
        falsification_rule_defined=True,
        notes="Le champ timelike n'a pas encore une origine physique unique dérivée.",
    ),
]

for claim in THEORY_REGISTRY:
    claim.validate()

theory_registry_df = pd.DataFrame([asdict(claim) for claim in THEORY_REGISTRY])
theory_registry_df


,claim_id,domain,observable,mathematical_expression,status,source_notebook,source_type,independent_of_test_data,parameters_fixed_before_test,gr_reference_defined,uncertainty_model_defined,falsification_rule_defined,notes
0,PPN_MAPPING_GVH,weak_field,PPN_gamma_beta,gamma_GVH=b1/a1 ; beta_GVH=a2/a1^2,DERIVED_COMPATIBILITY,GVH_Diagonal_Cubic_0.2.23.x,Article II / weak-field chain,True,False,True,False,False,Carte formelle vers PPN. Aucune valeur numériq...
1,STF_SOURCE_PROJECTOR,weak_field,anisotropic_stress_projection,pi_mn=P_mn^ab T_ab,DERIVED,GVH_Diagonal_Cubic_0.2.23.2,Article II / covariant source projector,True,True,False,False,False,"Objet mathématique utile, mais pas à lui seul ..."
2,TIMELIKE_FIELD_ORIGIN,weak_field,u_mu_physical_origin,NOT_UNIQUELY_DERIVED,HYPOTHESIS,GVH_Diagonal_Cubic_0.2.23.5.1,Article II / physical-origin audit,True,False,False,False,True,Le champ timelike n'a pas encore une origine p...


# 5. Porte THEORY_READY

Cette porte répond à une question différente de `DATA_READY` :

> **GVH fournit-il réellement une prédiction indépendante, définie avant le test, avec référence GR, incertitude et règle de falsification ?**

Une simple relation algébrique, un fit ou une hypothèse ne passe pas cette porte.


In [9]:
def theory_readiness_gate(claim: TheoryClaim):
    checks = {
        "status_is_testable_gvh": claim.status == "TESTABLE_GVH",
        "independent_of_test_data": bool(claim.independent_of_test_data),
        "parameters_fixed_before_test": bool(claim.parameters_fixed_before_test),
        "gr_reference_defined": bool(claim.gr_reference_defined),
        "uncertainty_model_defined": bool(claim.uncertainty_model_defined),
        "falsification_rule_defined": bool(claim.falsification_rule_defined),
    }
    checks["THEORY_READY"] = all(checks.values())
    return checks


theory_gate_rows = []

for claim in THEORY_REGISTRY:
    theory_gate_rows.append({
        "claim_id": claim.claim_id,
        "domain": claim.domain,
        "observable": claim.observable,
        "status": claim.status,
        **theory_readiness_gate(claim),
    })

theory_gate_df = pd.DataFrame(theory_gate_rows)
theory_gate_df


,claim_id,domain,observable,status,status_is_testable_gvh,independent_of_test_data,parameters_fixed_before_test,gr_reference_defined,uncertainty_model_defined,falsification_rule_defined,THEORY_READY
0,PPN_MAPPING_GVH,weak_field,PPN_gamma_beta,DERIVED_COMPATIBILITY,False,True,False,True,False,False,False
1,STF_SOURCE_PROJECTOR,weak_field,anisotropic_stress_projection,DERIVED,False,True,True,False,False,False,False
2,TIMELIKE_FIELD_ORIGIN,weak_field,u_mu_physical_origin,HYPOTHESIS,False,True,False,False,False,True,False


# 6. Porte combinée READY_FOR_GVH_TEST

Le test final est autorisé uniquement si les deux côtés du pont sont prêts :

\[
\boxed{
\mathrm{READY\_FOR\_GVH\_TEST}
=
\mathrm{DATA\_READY}
\land
\mathrm{THEORY\_READY}
}
\]

Cette logique évite trois erreurs :

1. tester une hypothèse comme si elle était une prédiction ;
2. ajuster les paramètres sur les mêmes données utilisées ensuite pour revendiquer un succès ;
3. confondre compatibilité avec GR et prédiction nouvelle de GVH.


In [10]:
CLAIM_DATA_REQUIREMENTS = {
    "PPN_MAPPING_GVH": ["PPN_SOLAR_SYSTEM"],
    "STF_SOURCE_PROJECTOR": ["PPN_SOLAR_SYSTEM"],
    "TIMELIKE_FIELD_ORIGIN": [],
}


def evaluate_combined_gate(claim_id):
    claim_row = theory_gate_df.loc[theory_gate_df["claim_id"] == claim_id]
    if claim_row.empty:
        raise KeyError(claim_id)

    theory_ready = bool(claim_row.iloc[0]["THEORY_READY"])
    required = CLAIM_DATA_REQUIREMENTS.get(claim_id, [])

    if required:
        subset = availability_df[availability_df["dataset_id"].isin(required)]
        data_ready = (
            len(subset) == len(required)
            and bool(subset["DATA_READY"].all())
        )
    else:
        data_ready = False

    ready_for_test = bool(data_ready and theory_ready)

    if ready_for_test:
        scientific_status = "READY_FOR_GVH_TEST"
    elif data_ready and not theory_ready:
        scientific_status = "DATA_READY_THEORY_BLOCKED"
    elif theory_ready and not data_ready:
        scientific_status = "THEORY_READY_DATA_BLOCKED"
    else:
        scientific_status = "INCONCLUSIVE"

    return {
        "claim_id": claim_id,
        "required_dataset_ids": required,
        "DATA_READY": data_ready,
        "THEORY_READY": theory_ready,
        "READY_FOR_GVH_TEST": ready_for_test,
        "scientific_status": scientific_status,
    }


combined_gate_df = pd.DataFrame([
    evaluate_combined_gate(claim_id)
    for claim_id in CLAIM_DATA_REQUIREMENTS
])

combined_gate_df


,claim_id,required_dataset_ids,DATA_READY,THEORY_READY,READY_FOR_GVH_TEST,scientific_status
0,PPN_MAPPING_GVH,[PPN_SOLAR_SYSTEM],False,False,False,INCONCLUSIVE
1,STF_SOURCE_PROJECTOR,[PPN_SOLAR_SYSTEM],False,False,False,INCONCLUSIVE
2,TIMELIKE_FIELD_ORIGIN,[],False,False,False,INCONCLUSIVE


# 7. Garde-fous anti-circularité

Une prédiction `TESTABLE_GVH` doit être enregistrée **avant** d’ouvrir ou d’utiliser les données qui serviront à la tester.

Le registre ci-dessous permet de tracer cette séparation.


In [11]:
PRE_REGISTRATION_COLUMNS = [
    "claim_id",
    "registration_utc",
    "prediction_expression",
    "parameter_values_json",
    "test_dataset_ids_json",
    "falsification_rule",
    "theory_source",
    "frozen_before_test",
]

pre_registration_df = pd.DataFrame(columns=PRE_REGISTRATION_COLUMNS)


def register_testable_prediction(
    registry,
    claim_id,
    prediction_expression,
    parameter_values,
    test_dataset_ids,
    falsification_rule,
    theory_source,
):
    claim = next((c for c in THEORY_REGISTRY if c.claim_id == claim_id), None)
    if claim is None:
        raise KeyError(claim_id)
    if claim.status != "TESTABLE_GVH":
        raise ValueError(
            f"{claim_id} n'est pas classé TESTABLE_GVH; pré-enregistrement interdit."
        )

    row = {
        "claim_id": claim_id,
        "registration_utc": datetime.now(timezone.utc).isoformat(),
        "prediction_expression": prediction_expression,
        "parameter_values_json": json.dumps(parameter_values, sort_keys=True),
        "test_dataset_ids_json": json.dumps(test_dataset_ids, sort_keys=True),
        "falsification_rule": falsification_rule,
        "theory_source": theory_source,
        "frozen_before_test": True,
    }
    return pd.concat([registry, pd.DataFrame([row])], ignore_index=True)


pre_registration_df


,claim_id,registration_utc,prediction_expression,parameter_values_json,test_dataset_ids_json,falsification_rule,theory_source,frozen_before_test


# 8. Domain gates de la Partie C

Les portes de domaine sont conservées, mais leur signification est désormais strictement **data-side**. Elles ne suffisent pas à déclarer qu’un test GVH est possible.


In [12]:
DOMAIN_GATES = {
    "neutron_star_eos": ["EOS_NUCLEAR_TABLES"],
    "neutron_star_structure": ["NICER_MASS_RADIUS", "MASSIVE_PULSARS"],
    "gravitational_waves": ["GW_PUBLIC_STRAIN", "GW_EVENT_POSTERIORS"],
    "weak_field": ["PPN_SOLAR_SYSTEM"],
    "cosmology": ["PANTHEON_PLUS", "BAO_COMPILATION", "HUBBLE_HZ", "CMB_PRODUCTS"],
}


def evaluate_domain_data_gate(domain):
    required = DOMAIN_GATES[domain]
    subset = availability_df[availability_df["dataset_id"].isin(required)]
    return {
        "domain": domain,
        "required_dataset_ids": required,
        "data_ready_dataset_ids": subset.loc[
            subset["DATA_READY"], "dataset_id"
        ].tolist(),
        "DATA_DOMAIN_READY": (
            len(subset) == len(required)
            and bool(subset["DATA_READY"].all())
        ),
    }


domain_gate_df = pd.DataFrame([
    evaluate_domain_data_gate(domain)
    for domain in DOMAIN_GATES
])

domain_gate_df


,domain,required_dataset_ids,data_ready_dataset_ids,DATA_DOMAIN_READY
0,neutron_star_eos,[EOS_NUCLEAR_TABLES],[],False
1,neutron_star_structure,"[NICER_MASS_RADIUS, MASSIVE_PULSARS]",[],False
2,gravitational_waves,"[GW_PUBLIC_STRAIN, GW_EVENT_POSTERIORS]",[],False
3,weak_field,[PPN_SOLAR_SYSTEM],[],False
4,cosmology,"[PANTHEON_PLUS, BAO_COMPILATION, HUBBLE_HZ, CM...",[],False


# 9. Empreinte logique et reproductibilité

L’empreinte SHA-256 dépend du contenu logique des cellules, pas des compteurs d’exécution.


In [13]:
def logical_notebook_fingerprint(notebook_path):
    notebook_path = Path(notebook_path)

    if not notebook_path.exists():
        return "NOT_AVAILABLE"

    with notebook_path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    logical_payload = {
        "cells": [
            {
                "cell_type": cell.get("cell_type"),
                "source": cell.get("source", []),
            }
            for cell in notebook.get("cells", [])
        ],
        "kernelspec": notebook.get("metadata", {}).get("kernelspec", {}),
        "language_info": notebook.get("metadata", {}).get("language_info", {}),
    }

    encoded = json.dumps(
        logical_payload,
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


CURRENT_NOTEBOOK_CANDIDATES = [
    Path("/content/GVH_Diagonal_Cubic_0.3.1.2_Real_Observation_Data_Framework_Theory_Data_Gates.ipynb"),
    Path("GVH_Diagonal_Cubic_0.3.1.2_Real_Observation_Data_Framework_Theory_Data_Gates.ipynb"),
]

CURRENT_NOTEBOOK_PATH = next(
    (path for path in CURRENT_NOTEBOOK_CANDIDATES if path.exists()),
    None,
)

NOTEBOOK_SHA256 = (
    logical_notebook_fingerprint(CURRENT_NOTEBOOK_PATH)
    if CURRENT_NOTEBOOK_PATH is not None
    else "NOT_AVAILABLE_IN_CURRENT_RUNTIME"
)

reproducibility_df = pd.DataFrame([{
    **ENVIRONMENT_INFO,
    "notebook_path": (
        str(CURRENT_NOTEBOOK_PATH)
        if CURRENT_NOTEBOOK_PATH is not None
        else "NOT_AVAILABLE"
    ),
    "notebook_sha256": NOTEBOOK_SHA256,
}])

reproducibility_df


,notebook_id,notebook_version,part_c_version,framework_name,author,execution_utc,python_version,python_executable,platform,operating_system,numpy_version,pandas_version,notebook_path,notebook_sha256
0,GVH_Diagonal_Cubic_0.3.1.2,0.3.1.2,0.2C,GVH Diagonal Cubic,Charlemagne O Laurince,2026-08-07T23:20:32.915136+00:00,"3.12.13 (main, Mar 4 2026, 09:23:07) [GCC 11....",/usr/bin/python3,Linux-6.6.122+-x86_64-with-glibc2.35,posix,2.0.2,2.2.2,NOT_AVAILABLE,NOT_AVAILABLE_IN_CURRENT_RUNTIME


# 10. Validation automatique

Cette validation porte sur l’infrastructure et les garde-fous. Elle ne doit jamais être lue comme une validation physique de GVH.


In [14]:
validation_rows = [
    {
        "test": "notebook identity",
        "status": "PASS"
        if NOTEBOOK_ID == "GVH_Diagonal_Cubic_0.3.1.2"
        and NOTEBOOK_VERSION == "0.3.1.2"
        else "FAIL",
    },
    {
        "test": "registry nonempty",
        "status": "PASS" if len(DATASET_REGISTRY) > 0 else "FAIL",
    },
    {
        "test": "unique dataset ids",
        "status": "PASS" if registry_df["dataset_id"].is_unique else "FAIL",
    },
    {
        "test": "valid data statuses",
        "status": "PASS"
        if registry_df["status"].isin(ALLOWED_DATA_STATUSES).all()
        else "FAIL",
    },
    {
        "test": "theory registry nonempty",
        "status": "PASS" if len(THEORY_REGISTRY) > 0 else "FAIL",
    },
    {
        "test": "unique theory claim ids",
        "status": "PASS"
        if theory_registry_df["claim_id"].is_unique
        else "FAIL",
    },
    {
        "test": "valid theory statuses",
        "status": "PASS"
        if theory_registry_df["status"].isin(ALLOWED_THEORY_STATUSES).all()
        else "FAIL",
    },
    {
        "test": "DATA_READY and THEORY_READY are distinct",
        "status": "PASS"
        if "DATA_READY" in availability_df.columns
        and "THEORY_READY" in theory_gate_df.columns
        else "FAIL",
    },
    {
        "test": "combined GVH gate exists",
        "status": "PASS"
        if "READY_FOR_GVH_TEST" in combined_gate_df.columns
        else "FAIL",
    },
    {
        "test": "no current claim falsely passes as TESTABLE_GVH",
        "status": "PASS"
        if not theory_registry_df["status"].eq("TESTABLE_GVH").any()
        and not combined_gate_df["READY_FOR_GVH_TEST"].any()
        else "FAIL",
    },
]

validation_df = pd.DataFrame(validation_rows)

INFRASTRUCTURE_PASS = bool((validation_df["status"] == "PASS").all())

if INFRASTRUCTURE_PASS and not combined_gate_df["READY_FOR_GVH_TEST"].any():
    OVERALL_STATUS = "PASS-FRAMEWORK-THEORY-DATA-GATES-GVH-TEST-BLOCKED"
elif INFRASTRUCTURE_PASS:
    OVERALL_STATUS = "PASS-FRAMEWORK-AT-LEAST-ONE-GVH-TEST-READY"
else:
    OVERALL_STATUS = "FAIL-FRAMEWORK-VALIDATION"

validation_df, OVERALL_STATUS


(                                              test status
 0                                notebook identity   PASS
 1                                registry nonempty   PASS
 2                               unique dataset ids   PASS
 3                              valid data statuses   PASS
 4                         theory registry nonempty   PASS
 5                          unique theory claim ids   PASS
 6                            valid theory statuses   PASS
 7         DATA_READY and THEORY_READY are distinct   PASS
 8                         combined GVH gate exists   PASS
 9  no current claim falsely passes as TESTABLE_GVH   PASS,
 'PASS-FRAMEWORK-THEORY-DATA-GATES-GVH-TEST-BLOCKED')

# 11. Cartographie officielle de la Partie C

L’architecture initiale C1 → C7 reste la structure scientifique officielle.

```text
0.2C_physical_predictions_and_falsification/

├── 0.2C1_prediction_foundations/
├── 0.2C2_weak_field_predictions/
├── 0.2C3_solar_system_tests/
├── 0.2C4_PPN_constraints/
├── 0.2C5_strong_field_predictions/
├── 0.2C6_observational_falsification/
└── 0.2C7_model_comparison/
```

Le framework 0.3.1.2 est une infrastructure transversale qui alimente ces sous-dossiers. Il ne remplace pas cette architecture.


In [15]:
PART_C_MAP = pd.DataFrame([
    {
        "folder": "0.2C1_prediction_foundations",
        "goal": "Définir les observables et classifier DERIVED/HYPOTHESIS/FIT/REFERENCE_GR/TESTABLE_GVH",
        "gate": "THEORY_READY",
        "priority": "P0",
    },
    {
        "folder": "0.2C2_weak_field_predictions",
        "goal": "Dériver les corrections weak-field sans utiliser les données de test pour fixer leur forme",
        "gate": "THEORY_READY",
        "priority": "P0",
    },
    {
        "folder": "0.2C3_solar_system_tests",
        "goal": "Périhélie, lumière, Shapiro, redshift et autres observables du système solaire",
        "gate": "READY_FOR_GVH_TEST",
        "priority": "P0",
    },
    {
        "folder": "0.2C4_PPN_constraints",
        "goal": "Confronter gamma, beta et autres paramètres aux contraintes observationnelles",
        "gate": "READY_FOR_GVH_TEST",
        "priority": "P0",
    },
    {
        "folder": "0.2C5_strong_field_predictions",
        "goal": "Objets compacts et champ fort",
        "gate": "READY_FOR_GVH_TEST",
        "priority": "P1",
    },
    {
        "folder": "0.2C6_observational_falsification",
        "goal": "Appliquer des critères pouvant réellement réfuter une formulation GVH",
        "gate": "READY_FOR_GVH_TEST",
        "priority": "P1",
    },
    {
        "folder": "0.2C7_model_comparison",
        "goal": "Comparer GVH à GR statistiquement sans circularité",
        "gate": "READY_FOR_GVH_TEST",
        "priority": "P1",
    },
])

PART_C_MAP


,folder,goal,gate,priority
0,0.2C1_prediction_foundations,Définir les observables et classifier DERIVED/...,THEORY_READY,P0
1,0.2C2_weak_field_predictions,Dériver les corrections weak-field sans utilis...,THEORY_READY,P0
2,0.2C3_solar_system_tests,"Périhélie, lumière, Shapiro, redshift et autre...",READY_FOR_GVH_TEST,P0
3,0.2C4_PPN_constraints,"Confronter gamma, beta et autres paramètres au...",READY_FOR_GVH_TEST,P0
4,0.2C5_strong_field_predictions,Objets compacts et champ fort,READY_FOR_GVH_TEST,P1
5,0.2C6_observational_falsification,Appliquer des critères pouvant réellement réfu...,READY_FOR_GVH_TEST,P1
6,0.2C7_model_comparison,Comparer GVH à GR statistiquement sans circula...,READY_FOR_GVH_TEST,P1


# 12. Export

Les exports rendent explicites la maturité des données, la maturité théorique et le statut combiné des tests.


In [16]:
EXPORT_PREFIX = "GVH_Diagonal_Cubic_0.3.1.2"

registry_export_df = registry_df.copy()
for column in ["expected_columns", "expected_units"]:
    registry_export_df[column] = registry_export_df[column].apply(
        lambda value: json.dumps(value, ensure_ascii=False, sort_keys=True)
    )

registry_export_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Dataset_Registry.csv",
    index=False,
)
availability_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Data_Readiness.csv",
    index=False,
)
metadata_audit_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Metadata_Audit.csv",
    index=False,
)
theory_registry_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Theory_Registry.csv",
    index=False,
)
theory_gate_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Theory_Readiness.csv",
    index=False,
)
combined_gate_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Combined_GVH_Test_Gates.csv",
    index=False,
)
domain_gate_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Domain_Data_Gates.csv",
    index=False,
)
pre_registration_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Prediction_PreRegistration.csv",
    index=False,
)
reproducibility_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Reproducibility.csv",
    index=False,
)
validation_df.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Validation.csv",
    index=False,
)
PART_C_MAP.to_csv(
    EXPORT_DIR / f"{EXPORT_PREFIX}_Part_C_Map.csv",
    index=False,
)

framework_metadata = {
    **ENVIRONMENT_INFO,
    "overall_status": OVERALL_STATUS,
    "notebook_sha256": NOTEBOOK_SHA256,
    "dataset_registry_count": int(len(DATASET_REGISTRY)),
    "theory_claim_count": int(len(THEORY_REGISTRY)),
    "data_ready_count": int(availability_df["DATA_READY"].sum()),
    "theory_ready_count": int(theory_gate_df["THEORY_READY"].sum()),
    "ready_for_gvh_test_count": int(combined_gate_df["READY_FOR_GVH_TEST"].sum()),
}

with (EXPORT_DIR / f"{EXPORT_PREFIX}_Framework_Metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(framework_metadata, file, ensure_ascii=False, indent=2)

framework_metadata


{'notebook_id': 'GVH_Diagonal_Cubic_0.3.1.2',
 'notebook_version': '0.3.1.2',
 'part_c_version': '0.2C',
 'framework_name': 'GVH Diagonal Cubic',
 'author': 'Charlemagne O Laurince',
 'execution_utc': '2026-08-07T23:20:32.915136+00:00',
 'python_version': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]',
 'python_executable': '/usr/bin/python3',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
 'operating_system': 'posix',
 'numpy_version': '2.0.2',
 'pandas_version': '2.2.2',
 'overall_status': 'PASS-FRAMEWORK-THEORY-DATA-GATES-GVH-TEST-BLOCKED',
 'notebook_sha256': 'NOT_AVAILABLE_IN_CURRENT_RUNTIME',
 'dataset_registry_count': 10,
 'theory_claim_count': 3,
 'data_ready_count': 0,
 'theory_ready_count': 0,
 'ready_for_gvh_test_count': 0}

# Conclusion

La version **0.3.1.2** consolide le framework de données réelles après les Articles I–II.

Elle impose désormais la chaîne :

\[
\boxed{
\text{donnée vérifiée}
\rightarrow
\mathrm{DATA\_READY}
}
\]

en parallèle avec :

\[
\boxed{
\text{prédiction GVH dérivée et pré-enregistrée}
\rightarrow
\mathrm{THEORY\_READY}
}
\]

puis seulement :

\[
\boxed{
\mathrm{DATA\_READY}
\land
\mathrm{THEORY\_READY}
\rightarrow
\mathrm{READY\_FOR\_GVH\_TEST}
}
\]

## Interprétation du statut attendu au stade actuel

Avec les claims hérités des Articles I–II, aucun objet n’est encore automatiquement promu en `TESTABLE_GVH`.

Le statut sain attendu est donc typiquement :

```text
PASS-FRAMEWORK-THEORY-DATA-GATES-GVH-TEST-BLOCKED
```

Cela signifie :

- **PASS** : l’infrastructure et les garde-fous fonctionnent ;
- **GVH-TEST-BLOCKED** : aucune prédiction indépendante n’a encore satisfait toutes les conditions permettant un test physique.

Ce résultat n’est pas un échec de GVH. C’est une protection contre une conclusion prématurée.

## Étape suivante

Avant de lancer un nouveau domaine observationnel, utiliser **0.2C1_prediction_foundations** pour déterminer si une prédiction candidate peut être classée `TESTABLE_GVH`.

Ensuite seulement, activer les tests de C2–C7 correspondants.
